# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, not a dict

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets and list the record set, field, and column `@id` values. All entities are referenced by their `@id`.

In [ ]:
# List all record sets, with their @id and field @id's
print("Available record sets in this dataset:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for field in fields:
        if hasattr(field, 'get'):
            field_id = field.get('@id', str(field))
        else:
            field_id = str(field)
        print(f"    Field @id: {field_id}")
        # Each field may refer to a column (for tabular)
        column = field.get('column') if hasattr(field, 'get') else None
        if column:
            if isinstance(column, list):
                for col in column:
                    print(f"        Column @id: {col.get('@id', str(col))}")
            else:
                print(f"        Column @id: {column.get('@id', str(column))}")
if len(record_sets) == 0:
    print("No record sets defined explicitly in the schema; data may be available through default or implicit record sets.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

_Since this dataset may not have explicit `recordSet` definitions in the top-level schema, we will auto-detect available record set IDs for extraction._

In [ ]:
# If record sets are not specified, mlcroissant exposes defaults based on DataFiles (distributions)
if len(record_sets) == 0:
    # Try to list data file @id's
    print("Dataset has no explicit record sets; listing available data file record sets.")
    records_set_ids = [rs['@id'] for rs in dataset.record_sets]
else:
    records_set_ids = [rs['@id'] for rs in record_sets]

print("Record set @id's found:")
for rid in records_set_ids:
    print(f"- {rid}")

# Use all available record sets for demo
dataframes = {}
for record_set_id in records_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print(f"No records loaded for record set {record_set_id}")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set: {record_set_id}")
        print("Example columns:", df.columns.tolist())
        display(df.head())
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")
        continue

# Pick the first available record set with data for further analysis
if len(dataframes):
    first_record_set_id = list(dataframes.keys())[0]
    print(f"Using record set {first_record_set_id} for analysis.")
else:
    raise ValueError("No dataframes loaded from any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on a numeric field, normalization, and grouping.

Use only `@id` to reference fields (column names), as per the dataset schema.

In [ ]:
# Choose the analysis DataFrame
df = dataframes[first_record_set_id].copy()
print(f"Columns in use (field @id's): {df.columns.tolist()}")

# Attempt to automatically pick a numeric field for demo
# We'll select a field whose dtype is float or int and is not an index/ID field
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) and ('id' not in col.lower())]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Selected numeric field for analysis: {numeric_field_id}")
else:
    # Fallback to any column
    numeric_field_id = df.columns[0]
    print(f"Selected first available column for analysis: {numeric_field_id}")

# Set threshold (mean or 75th percentile for demo)
try:
    threshold = df[numeric_field_id].quantile(0.75) if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else None
except Exception:
    threshold = None

if threshold is not None:
    filtered_df = df[df[numeric_field_id] > threshold]
else:
    filtered_df = df.copy()

print(f"Filtered records with {numeric_field_id} > {threshold} (if applicable):")
display(filtered_df.head())

# Normalization (z-score)
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized numeric field: {numeric_field_id}")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field (categorical or string type)
group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
if group_fields:
    group_field_id = group_fields[0]
    print(f"Grouping by: {group_field_id}")
    try:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        display(grouped_df.head())
    except Exception as e:
        print(f"Could not group by {group_field_id}: {e}")
else:
    print("No suitable non-numeric group field found.")

## 5. Visualization
Visualize numeric field distributions and relationships (using field `@id`s).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field histogram
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], kde=True, bins=20, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# If grouping field exists, show boxplot
if 'group_field_id' in locals() and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(10,6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to explore the FAIR^2 dataset using the `mlcroissant` library:
- Loaded metadata and reviewed available record sets and fields by their `@id`
- Loaded tabular data from a discovered record set
- Performed preliminary EDA (filter, normalize, group)
- Visualized numeric distributions and relationships

To extend your analysis, refer to the record set and field `@id` references and try exploring additional attributes or using domain context (e.g., socio-demographic or regression results) from the dataset schema.